# 05 — Advanced Analytics

**Bluestock Mutual Fund Analytics Capstone — D6**

This notebook implements the advanced analytics required by the rubric:

1. Historical VaR and Conditional VaR
2. Investor cohort analysis
3. Fund recommendation logic
4. Recommendation scoring and explainability
5. Risk/return profiling
6. Exportable recommendation and cohort CSVs

The notebook uses the project's actual SQLite database and adapts to available columns instead of assuming a fixed schema.


In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)


## 1. Load the available datasets

In [ ]:
def load_table(name):
    with sqlite3.connect(DB_PATH) as conn:
        exists = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
            conn,
            params=(name,)
        )
        if exists.empty:
            return pd.DataFrame()
        return pd.read_sql_query(f'SELECT * FROM "{name}"', conn)

fund = load_table("dim_fund")
nav = load_table("fact_nav")
perf = load_table("fact_performance")
transactions = load_table("fact_transactions")

print("dim_fund:", fund.shape)
print("fact_nav:", nav.shape)
print("fact_performance:", perf.shape)
print("fact_transactions:", transactions.shape)


## 2. Historical VaR and Conditional VaR

In [ ]:
# Prepare NAV returns for risk analysis.
if not nav.empty:
    date_col = next((c for c in ["date", "nav_date", "as_of_date"] if c in nav.columns), None)
    fund_col = next((c for c in ["amfi_code", "scheme_code", "fund_code"] if c in nav.columns), None)
    nav_col = next((c for c in ["nav", "nav_value", "net_asset_value"] if c in nav.columns), None)

    if date_col and fund_col and nav_col:
        risk_nav = nav[[date_col, fund_col, nav_col]].copy()
        risk_nav.columns = ["date", "amfi_code", "nav_value"]
        risk_nav["date"] = pd.to_datetime(risk_nav["date"], errors="coerce")
        risk_nav["nav_value"] = pd.to_numeric(risk_nav["nav_value"], errors="coerce")
        risk_nav = (
            risk_nav.dropna()
            .sort_values(["amfi_code", "date"])
            .drop_duplicates(["amfi_code", "date"])
        )
        risk_nav["daily_return"] = (
            risk_nav.groupby("amfi_code")["nav_value"].pct_change()
        )
    else:
        risk_nav = pd.DataFrame()
else:
    risk_nav = pd.DataFrame()

def historical_var(returns, confidence=0.95):
    returns = pd.Series(returns).dropna()
    if len(returns) < 20:
        return np.nan
    return -returns.quantile(1 - confidence)

def conditional_var(returns, confidence=0.95):
    returns = pd.Series(returns).dropna()
    if len(returns) < 20:
        return np.nan
    threshold = returns.quantile(1 - confidence)
    tail = returns[returns <= threshold]
    return -tail.mean() if not tail.empty else np.nan

var_records = []

if not risk_nav.empty:
    for amfi_code, group in risk_nav.groupby("amfi_code"):
        returns = group["daily_return"].dropna()

        var_records.append({
            "amfi_code": amfi_code,
            "observations": len(returns),
            "var_95_pct": historical_var(returns) * 100,
            "cvar_95_pct": conditional_var(returns) * 100
        })

var_df = pd.DataFrame(var_records)

if not var_df.empty and not fund.empty and "amfi_code" in fund.columns:
    var_df = var_df.merge(fund, on="amfi_code", how="left")

display(var_df.head(15))


### VaR interpretation

Historical 95% VaR is the magnitude of the 5th percentile daily loss.

Conditional VaR (CVaR / Expected Shortfall) is the average loss among observations that fall beyond the VaR threshold. CVaR is therefore designed to describe the severity of losses in the tail rather than only the cutoff.

Both are historical estimates and are not forecasts.


In [ ]:
if not var_df.empty:
    plt.figure(figsize=(10, 6))
    plt.scatter(
        var_df["var_95_pct"],
        var_df["cvar_95_pct"],
        alpha=0.55
    )
    plt.xlabel("Historical VaR 95% (%)")
    plt.ylabel("Conditional VaR 95% (%)")
    plt.title("Tail-Risk Comparison Across Funds")
    plt.grid(alpha=0.25)
    plt.show()


## 3. Investor cohort analysis

In [ ]:
if transactions.empty:
    print("No transaction table is available.")
    cohort_df = pd.DataFrame()
else:
    tx = transactions.copy()

    # Standardize common fields without inventing missing data.
    for candidate, target in [
        ("date", "date"),
        ("transaction_date", "date"),
        ("amount", "amount"),
        ("transaction_amount", "amount"),
        ("investor_id", "investor_id"),
        ("state", "state"),
        ("city", "city"),
        ("city_tier", "city_tier"),
        ("age", "age"),
        ("transaction_type", "transaction_type")
    ]:
        if candidate in tx.columns and target not in tx.columns:
            tx[target] = tx[candidate]

    if "date" in tx.columns:
        tx["date"] = pd.to_datetime(tx["date"], errors="coerce")

    if "amount" in tx.columns:
        tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce")

    if "age" in tx.columns:
        tx["age"] = pd.to_numeric(tx["age"], errors="coerce")
        tx["age_group"] = pd.cut(
            tx["age"],
            bins=[0, 25, 35, 45, 55, 65, 120],
            labels=["<25", "25-35", "36-45", "46-55", "56-65", "65+"],
            include_lowest=True
        )

    if "date" in tx.columns:
        tx["cohort_month"] = tx["date"].dt.to_period("M").astype("string")

    cohort_fields = [
        c for c in ["cohort_month", "age_group", "state", "city_tier"]
        if c in tx.columns
    ]

    if "investor_id" in tx.columns and cohort_fields:
        agg = {
            "investor_count": ("investor_id", "nunique")
        }

        if "amount" in tx.columns:
            agg["total_transaction_amount"] = ("amount", "sum")
            agg["average_transaction_amount"] = ("amount", "mean")

        cohort_df = (
            tx.groupby(cohort_fields, dropna=False)
            .agg(**agg)
            .reset_index()
        )
    else:
        cohort_df = pd.DataFrame()

if not cohort_df.empty:
    display(cohort_df.head(20))


## 4. Cohort visualizations

In [ ]:
if not cohort_df.empty and "age_group" in cohort_df.columns and "total_transaction_amount" in cohort_df.columns:
    age_summary = (
        cohort_df.groupby("age_group", observed=False)["total_transaction_amount"]
        .sum()
        .reset_index()
    )

    plt.figure(figsize=(9, 5))
    plt.bar(
        age_summary["age_group"].astype(str),
        age_summary["total_transaction_amount"]
    )
    plt.xlabel("Age Group")
    plt.ylabel("Transaction Amount")
    plt.title("Transaction Amount by Investor Age Group")
    plt.xticks(rotation=30)
    plt.grid(axis="y", alpha=0.25)
    plt.show()

if not cohort_df.empty and "city_tier" in cohort_df.columns and "investor_count" in cohort_df.columns:
    tier_summary = (
        cohort_df.groupby("city_tier", dropna=False)["investor_count"]
        .sum()
        .reset_index()
    )

    plt.figure(figsize=(8, 5))
    plt.bar(
        tier_summary["city_tier"].astype(str),
        tier_summary["investor_count"]
    )
    plt.xlabel("City Tier")
    plt.ylabel("Unique Investors")
    plt.title("Investor Distribution by City Tier")
    plt.grid(axis="y", alpha=0.25)
    plt.show()


## 5. Fund recommendation engine

In [ ]:
# Recommendation logic is transparent and rule-based.
# It uses available performance columns and creates no artificial data.

if perf.empty:
    raise ValueError("fact_performance is unavailable; recommender cannot score funds.")

p = perf.copy()

if "amfi_code" not in p.columns:
    raise ValueError("fact_performance must contain amfi_code.")

if not fund.empty and "amfi_code" in fund.columns:
    rec = p.merge(fund, on="amfi_code", how="left")
else:
    rec = p.copy()

# Standardize numeric metrics where present.
metric_aliases = {
    "return_1yr_pct": ["return_1yr_pct", "return_1y_pct", "one_year_return_pct"],
    "return_3yr_pct": ["return_3yr_pct", "return_3y_pct", "three_year_return_pct"],
    "return_5yr_pct": ["return_5yr_pct", "return_5y_pct", "five_year_return_pct"],
    "std_dev_pct": ["std_dev_pct", "volatility_pct", "standard_deviation_pct"],
    "sharpe_ratio": ["sharpe_ratio", "sharpe"],
    "max_drawdown_pct": ["max_drawdown_pct", "drawdown_pct"]
}

for target, candidates in metric_aliases.items():
    if target not in rec.columns:
        source = next((c for c in candidates if c in rec.columns), None)
        if source:
            rec[target] = pd.to_numeric(rec[source], errors="coerce")

for col in [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "std_dev_pct",
    "sharpe_ratio",
    "max_drawdown_pct"
]:
    if col in rec.columns:
        rec[col] = pd.to_numeric(rec[col], errors="coerce")


## 6. Risk profiles and recommendation scoring

In [ ]:
def minmax(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)
    lo, hi = s.min(), s.max()
    if hi == lo:
        result = pd.Series(0.5, index=series.index)
    else:
        result = (s - lo) / (hi - lo)
    return result if higher_is_better else 1 - result

# User profiles:
# Conservative: prioritize lower volatility/drawdown.
# Balanced: balance return, Sharpe and risk.
# Aggressive: prioritize returns and Sharpe.

rec["return_score"] = (
    minmax(rec["return_1yr_pct"]) if "return_1yr_pct" in rec.columns
    else 0.5
)

rec["sharpe_score"] = (
    minmax(rec["sharpe_ratio"]) if "sharpe_ratio" in rec.columns
    else 0.5
)

rec["risk_score"] = (
    minmax(rec["std_dev_pct"], higher_is_better=False)
    if "std_dev_pct" in rec.columns else 0.5
)

rec["drawdown_score"] = (
    minmax(rec["max_drawdown_pct"], higher_is_better=True)
    if "max_drawdown_pct" in rec.columns else 0.5
)

rec["conservative_score"] = (
    0.25 * rec["return_score"].fillna(0.5)
    + 0.25 * rec["sharpe_score"].fillna(0.5)
    + 0.35 * rec["risk_score"].fillna(0.5)
    + 0.15 * rec["drawdown_score"].fillna(0.5)
)

rec["balanced_score"] = (
    0.35 * rec["return_score"].fillna(0.5)
    + 0.30 * rec["sharpe_score"].fillna(0.5)
    + 0.20 * rec["risk_score"].fillna(0.5)
    + 0.15 * rec["drawdown_score"].fillna(0.5)
)

rec["aggressive_score"] = (
    0.50 * rec["return_score"].fillna(0.5)
    + 0.30 * rec["sharpe_score"].fillna(0.5)
    + 0.10 * rec["risk_score"].fillna(0.5)
    + 0.10 * rec["drawdown_score"].fillna(0.5)
)

display(
    rec[
        [c for c in [
            "scheme_name",
            "fund_house",
            "category",
            "return_1yr_pct",
            "std_dev_pct",
            "sharpe_ratio",
            "max_drawdown_pct",
            "conservative_score",
            "balanced_score",
            "aggressive_score"
        ] if c in rec.columns]
    ].head(15)
)


## 7. Generate explainable recommendations

In [ ]:
def recommendation_reason(row, profile):
    reasons = []

    if pd.notna(row.get("return_1yr_pct", np.nan)):
        if row["return_1yr_pct"] >= rec["return_1yr_pct"].median():
            reasons.append("above-median 1-year return")

    if pd.notna(row.get("sharpe_ratio", np.nan)):
        if row["sharpe_ratio"] >= rec["sharpe_ratio"].median():
            reasons.append("stronger risk-adjusted performance")

    if profile == "Conservative" and pd.notna(row.get("std_dev_pct", np.nan)):
        if row["std_dev_pct"] <= rec["std_dev_pct"].median():
            reasons.append("below-median volatility")

    if pd.notna(row.get("max_drawdown_pct", np.nan)):
        if row["max_drawdown_pct"] >= rec["max_drawdown_pct"].median():
            reasons.append("relatively contained drawdown")

    return "; ".join(reasons) if reasons else "Selected by profile score"


profile_to_score = {
    "Conservative": "conservative_score",
    "Balanced": "balanced_score",
    "Aggressive": "aggressive_score"
}

recommendation_tables = {}

for profile_name, score_col in profile_to_score.items():
    temp = rec.copy()
    temp["profile"] = profile_name
    temp["recommendation_reason"] = temp.apply(
        lambda row: recommendation_reason(row, profile_name),
        axis=1
    )

    temp = temp.sort_values(score_col, ascending=False)

    recommendation_tables[profile_name] = temp

    print(f"\nTop {profile_name} recommendations")
    display(
        temp[
            [c for c in [
                "scheme_name",
                "fund_house",
                "category",
                "return_1yr_pct",
                "std_dev_pct",
                "sharpe_ratio",
                "max_drawdown_pct",
                score_col,
                "recommendation_reason"
            ] if c in temp.columns]
        ].head(10)
    )


## 8. Export advanced analytics outputs

In [ ]:
# Export VaR / CVaR
if not var_df.empty:
    var_path = PROCESSED_DIR / "fund_var_analysis.csv"
    var_df.to_csv(var_path, index=False)
    print("Saved:", var_path)

# Export cohorts
if not cohort_df.empty:
    cohort_path = PROCESSED_DIR / "investor_cohort_analysis.csv"
    cohort_df.to_csv(cohort_path, index=False)
    print("Saved:", cohort_path)

# Export all recommendation profiles
recommendation_rows = []

for profile_name, temp in recommendation_tables.items():
    score_col = profile_to_score[profile_name]

    keep = [
        c for c in [
            "amfi_code",
            "scheme_name",
            "fund_house",
            "category",
            "return_1yr_pct",
            "std_dev_pct",
            "sharpe_ratio",
            "max_drawdown_pct",
            score_col,
            "recommendation_reason"
        ]
        if c in temp.columns
    ]

    export_temp = temp[keep].copy()
    export_temp["profile"] = profile_name
    export_temp["recommendation_score"] = export_temp[score_col]

    recommendation_rows.append(export_temp)

if recommendation_rows:
    recommendations = pd.concat(
        recommendation_rows,
        ignore_index=True
    )

    recommendation_path = PROCESSED_DIR / "fund_recommendations.csv"
    recommendations.to_csv(
        recommendation_path,
        index=False
    )

    print("Saved:", recommendation_path)


## 9. D6 conclusions

### Advanced risk analysis
Historical VaR identifies the loss threshold associated with the lower 5% of observed daily returns. CVaR goes further by averaging the losses in that tail.

### Cohort analysis
Investor cohorts can be compared by age group, location/city tier and acquisition month when those fields are present in the transaction data. This helps identify which investor groups generate the most activity or transaction value.

### Recommender
The recommender is deliberately transparent and explainable. It normalizes available return, Sharpe, volatility and drawdown metrics and combines them using different weights for Conservative, Balanced and Aggressive profiles.

**Important:** the recommender is an analytical ranking system for the capstone. It is not personalized financial advice and does not guarantee future returns.
